# Chapter 04: Inside Large Language Models

**Part:** Part II: Core Generative AI & Large Language Models  
**Chapter Identifier:** `ch04`  

## Core Concepts Covered in this Chapter:
- **Transformer Architecture**
- **Attention Mechanisms**
- **BPE Tokenization**
- **Positional Encoding**
- **Next-Token Prediction**
- **Instruction Tuning**
- **RLHF/DPO**
- **Context Window**
- **Temp/Top-k/p**
- **Hallucinations**
- **LoRA**
- **Quantization**
- **KV Cache**
- **SLMs**
- **Reasoning Models**

---


## 1. Tokenization (BPE) & Positional Encoding

Text is broken down into sub-word tokens using Byte-Pair Encoding (BPE). Because Transformers have no inherent recurrent ordering, Positional Encodings (sinusoidal or RoPE) are injected to provide sequence awareness.

```mermaid
flowchart LR
    A["Raw Text String"] --> B["BPE Tokenizer"]
    B --> C["Token IDs: [15496, 11, 284]"]
    C --> D["Token Embeddings + Positional Encodings"]
    D --> E["Transformer Decoder Blocks"]
```


In [ ]:
def toy_bpe_tokenize(text: str, vocab: dict) -> list:
    tokens = []
    i = 0
    while i < len(text):
        matched = False
        for length in range(min(12, len(text) - i), 0, -1):
            chunk = text[i:i+length]
            if chunk in vocab:
                tokens.append(vocab[chunk])
                i += length
                matched = True
                break
        if not matched:
            tokens.append(vocab.get("<UNK>", 0))
            i += 1
    return tokens

vocab = {"AI": 101, " Engineering": 102, "is": 103, " powerful": 104, "<UNK>": 0}
tokens = toy_bpe_tokenize("AI Engineering is powerful", vocab)
print(f"Token IDs: {tokens}")

## 2. Sampling Strategies: Temperature, Top-k, and Top-p (Nucleus)

During generation, LLMs produce unnormalized logit distributions over vocabulary tokens. Decoding parameters control creativity vs. determinism.


In [ ]:
import math

def sample_logits(logits: dict, temperature: float = 1.0, top_k: int = 3, top_p: float = 0.9) -> list:
    scaled = {k: v / max(temperature, 1e-4) for k, v in logits.items()}
    exp_vals = {k: math.exp(v) for k, v in scaled.items()}
    total = sum(exp_vals.values())
    probs = {k: v / total for k, v in exp_vals.items()}
    sorted_probs = sorted(probs.items(), key=lambda x: x[1], reverse=True)
    filtered = sorted_probs[:top_k]
    cum_sum = 0.0
    nucleus = []
    for token, p in filtered:
        nucleus.append((token, p))
        cum_sum += p
        if cum_sum >= top_p:
            break
    return nucleus

logits_sample = {"the": 4.5, "a": 3.8, "one": 3.2, "quantum": 1.1, "banana": -2.0}
print("Sampled Distribution (Temp=0.7, Top-K=3, Top-P=0.85):")
for token, prob in sample_logits(logits_sample, temperature=0.7, top_k=3, top_p=0.85):
    print(f"  Token '{token}': {prob * 100:.2f}%")

## 3. The KV Cache, LoRA & Quantization (AWQ/GGUF)

In autoregressive inference, caching previous Key and Value matrices (KV Cache) avoids quadratic re-computation. LoRA (Low-Rank Adaptation) decomposes weight updates W = W_0 + B*A with rank r << d, enabling efficient fine-tuning.


In [ ]:
def calculate_kv_cache_memory(batch_size: int, seq_len: int, num_layers: int, num_heads: int, head_dim: int, bytes_per_elem: int = 2) -> float:
    total_bytes = 2 * batch_size * num_layers * num_heads * seq_len * head_dim * bytes_per_elem
    return total_bytes / (1024 * 1024)

mb_4k = calculate_kv_cache_memory(batch_size=1, seq_len=4096, num_layers=32, num_heads=32, head_dim=128)
mb_128k = calculate_kv_cache_memory(batch_size=1, seq_len=131072, num_layers=32, num_heads=32, head_dim=128)

print(f"Llama-3-8B KV Cache at 4k context: {mb_4k:.2f} MB")
print(f"Llama-3-8B KV Cache at 128k context: {mb_128k:.2f} MB ({mb_128k/1024:.2f} GB)")

---
## Hands-On Capstone Lab: Build a Complete Autoregressive Sampler & LoRA Decomposition Simulator

### Objective:
Simulate token-by-token generation with KV caching and calculate parameter efficiency of a LoRA adapter matrix.

### Instructions:
1. Simulate next-token prediction with a KV cache length counter.
2. Calculate parameter count of base weight matrix (d_in x d_out).
3. Calculate parameter count of LoRA matrices A and B where rank r=8.


In [ ]:
d_in, d_out = 4096, 4096
lora_rank = 8

base_params = d_in * d_out
lora_params = (d_in * lora_rank) + (lora_rank * d_out)
reduction_pct = (1.0 - (lora_params / base_params)) * 100

print(f"Base Projection Matrix: {d_in} x {d_out} = {base_params:,} parameters")
print(f"LoRA Adapter (Rank={lora_rank}): {lora_params:,} parameters")
print(f"Trainable Parameter Reduction: {reduction_pct:.2f}% reduction!")

---
## Chapter Mastery Self-Check Quiz

Test your understanding of the concepts covered in this chapter.

### Question 1: What is the primary purpose of the KV (Key-Value) Cache during LLM inference?
- **A)** To store the user's password securely
- **B)** To avoid re-computing Key and Value representations for past tokens at each new token generation step
- **C)** To compress the weights of the model onto disk
- **D)** To remove harmful content from outputs

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) To avoid re-computing Key and Value representations for past tokens at each new token generation step

**Explanation:** Because generation is autoregressive, previous tokens do not change. Caching past Keys and Values reduces per-token computational complexity from O(N^2) to O(N).
</details>

### Question 2: Setting Temperature = 0 in LLM sampling results in which behavior?
- **A)** Maximum randomness and diversity
- **B)** Greedy decoding (always picking the token with the single highest probability)
- **C)** An infinite generation loop
- **D)** Model crash due to zero division

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) Greedy decoding (always picking the token with the single highest probability)

**Explanation:** As temperature approaches zero, probability mass concentrates entirely on the top logit, yielding deterministic greedy decoding.
</details>

### Question 3: In Low-Rank Adaptation (LoRA), what does the rank 'r' represent?
- **A)** The number of GPUs required for training
- **B)** The inner dimension of the low-rank update matrices A and B
- **C)** The context window length
- **D)** The number of attention heads

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) The inner dimension of the low-rank update matrices A and B

**Explanation:** LoRA approximates weight update delta W with B * A where A in R^(r x d) and B in R^(d x r), where rank r << d determines the capacity of the adapter.
</details>

### Question 4: Which quantization technique compresses 16-bit floating point weights into 4-bit integers while preserving activation accuracy?
- **A)** BPE Tokenization
- **B)** AWQ / GPTQ Quantization
- **C)** Softmax Normalization
- **D)** RoPE Embedding

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) AWQ / GPTQ Quantization

**Explanation:** AWQ (Activation-aware Weight Quantization) and GPTQ quantize model weights to 4 bits while protecting salient weight channels.
</details>
